# Gold Layer — Dimension: Customer (SCD Type 2)
## SalesFlow Data Lakehouse | Extra Challenge: Slowly Changing Dimensions

Implements **SCD Type 2** on `dim_customer`, preserving full history of
customer attribute changes using Delta Lake's MERGE operation.

**SCD Type 2 Logic:**
| Scenario | Action |
|---|---|
| Customer doesn't exist | INSERT new record |
| Customer exists + data changed | CLOSE old record → INSERT new record |
| Customer exists + data unchanged | Do nothing |

**History tracking columns:**
| Column | Type | Description |
|---|---|---|
| `effective_date` | date | When this version became active |
| `end_date` | date | When this version was superseded (`NULL` if current) |
| `is_current` | boolean | `TRUE` if this is the active version |

**Note:** A customer can have multiple rows — one per version of their data.
The current version always has `is_current=TRUE` and `end_date=NULL`.

In [0]:
%run ../04_Utils/common_functions

## 1. Prepare the Target Table
Create `dim_customer_scd2` with the additional SCD2 columns if it doesn't exist.
On first run this will be empty — all Silver customers will be inserted as new records.

In [0]:
%sql
-- Create the SCD2 target table if it doesn't exist
-- Schema matches dim_customer + the three SCD2 tracking columns
CREATE TABLE IF NOT EXISTS salesflow_dev.gold.dim_customer_scd2 (
    customer_key    STRING,
    customer_id     STRING,
    company_name    STRING,
    contact_name    STRING,
    country         STRING,
    city            STRING,
    region          STRING,
    phone           STRING,
    effective_date  DATE,
    end_date        DATE,       -- NULL means this is the current active record
    is_current      BOOLEAN
)
USING DELTA;

## 2. Prepare the Incoming (Source) Data
Read VALID customers from Silver and build the surrogate key.
This represents the latest known state of each customer.

In [0]:
from pyspark.sql.functions import current_date, col, lit

# Read only VALID customers from Silver — latest state
df_source = spark.table("salesflow_dev.silver.customers") \
                 .filter(col("data_quality_status") == "VALID")

# Select and rename to snake_case
df_source = df_source.select(
    col("CustomerID").alias("customer_id"),
    col("CompanyName").alias("company_name"),
    col("ContactName").alias("contact_name"),
    col("Country").alias("country"),
    col("City").alias("city"),
    col("Region").alias("region"),
    col("Phone").alias("phone")
)

# Add surrogate key
df_source = add_surrogate_key(df_source, "customer", ["customer_id"])

# Add SCD2 columns for incoming records — all are new/current versions
df_source = df_source \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date",       lit(None).cast("date")) \
    .withColumn("is_current",     lit(True))

print(f"Incoming records from Silver: {df_source.count()}")
display(df_source.limit(5))

## 3. Register Source as Temp View
Required to reference the source DataFrame inside the SQL MERGE statement.

In [0]:
# Register as temp view so it can be referenced in the MERGE SQL below
df_source.createOrReplaceTempView("dim_customer_scd2_source")

## 4. MERGE — SCD Type 2 Logic

Three scenarios handled in a single MERGE operation:

**Step 1 — MERGE:** handles inserts and detects changes  
**Step 2 — UPDATE:** closes old records where data has changed

Delta Lake doesn't support updating the *old* record and inserting a *new* one
in a single MERGE pass, so we use **two operations**:
1. `MERGE` to insert new records and detect unchanged ones
2. `UPDATE` to close the previous version of changed records

In [0]:
from pyspark.sql.functions import current_date, col, lit
from delta.tables import DeltaTable

# Load current state of target
target = DeltaTable.forName(spark, "salesflow_dev.gold.dim_customer_scd2")
df_target = target.toDF()

# Load source (already created in Cell 6)
df_source = spark.table("dim_customer_scd2_source")

# -------------------------------------------------------
# STEP 1: Identify new and changed records
# -------------------------------------------------------

# Get current active records from target
df_current = df_target.filter(col("is_current") == True)

# Join source with current target to classify each record
df_joined = df_source.join(
    df_current.select(
        col("customer_id").alias("t_customer_id"),
        col("company_name").alias("t_company_name"),
        col("contact_name").alias("t_contact_name"),
        col("country").alias("t_country"),
        col("city").alias("t_city"),
        col("region").alias("t_region"),
        col("phone").alias("t_phone")
    ),
    df_source["customer_id"] == col("t_customer_id"),
    how="left"
)

# New customers: no match in target
df_new = df_joined.filter(col("t_customer_id").isNull()) \
                  .select(df_source.columns)

# Changed customers: exists in target but attributes differ
df_changed = df_joined.filter(
    col("t_customer_id").isNotNull() & (
        (col("company_name") != col("t_company_name")) |
        (col("contact_name") != col("t_contact_name")) |
        (col("country")      != col("t_country"))      |
        (col("city")         != col("t_city"))         |
        (col("region")       != col("t_region"))       |
        (col("phone")        != col("t_phone"))
    )
).select(df_source.columns)

print(f"New customers    : {df_new.count()}")
print(f"Changed customers: {df_changed.count()}")

# -------------------------------------------------------
# STEP 2: Close outdated records for changed customers
# -------------------------------------------------------
if df_changed.count() > 0:
    changed_ids = [row["customer_id"] for row in df_changed.select("customer_id").collect()]

    target.update(
        condition=(
            col("customer_id").isin(changed_ids) &
            (col("is_current") == True)
        ),
        set={
            "is_current": lit(False),
            "end_date":   current_date()
        }
    )
    print(f"Closed {len(changed_ids)} outdated record(s)")

# -------------------------------------------------------
# STEP 3: Insert new versions + brand new customers
# -------------------------------------------------------
df_to_insert = df_new.union(df_changed)

if df_to_insert.count() > 0:
    df_to_insert.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("salesflow_dev.gold.dim_customer_scd2")
    print(f"Inserted {df_to_insert.count()} new record(s)")
else:
    print("No changes detected — nothing to insert")

In [0]:
%sql
-- STEP 2: CLOSE outdated records
-- Find current records in target that have a newer version (duplicate is_current=TRUE)
-- and mark them as expired
UPDATE salesflow_dev.gold.dim_customer_scd2 AS target
SET
    is_current = FALSE,
    end_date   = current_date() - INTERVAL 1 DAY
WHERE
    is_current = TRUE
    AND EXISTS (
        SELECT 1
        FROM salesflow_dev.gold.dim_customer_scd2 newer
        WHERE newer.customer_id    = target.customer_id
          AND newer.is_current     = TRUE
          AND newer.effective_date > target.effective_date
    );

## 5. Validation

In [0]:
%sql
-- Total records in the SCD2 table (current + historical)
SELECT
    is_current,
    COUNT(*) AS records
FROM salesflow_dev.gold.dim_customer_scd2
GROUP BY is_current;

In [0]:
%sql
-- Customers with more than one version (history exists)
SELECT
    customer_id,
    COUNT(*) AS versions
FROM salesflow_dev.gold.dim_customer_scd2
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY versions DESC;

In [0]:
%sql
-- Verify no customer has more than one is_current=TRUE record
-- Result must always be empty
SELECT
    customer_id,
    COUNT(*) AS current_versions
FROM salesflow_dev.gold.dim_customer_scd2
WHERE is_current = TRUE
GROUP BY customer_id
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Verify no customer has more than one is_current=TRUE record
-- Result must always be empty
SELECT
    customer_id,
    COUNT(*) AS current_versions
FROM salesflow_dev.gold.dim_customer_scd2
WHERE is_current = TRUE
GROUP BY customer_id
HAVING COUNT(*) > 1;